In [1]:
# 1. Load and prepare the data


import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from statsmodels.tsa.holtwinters import ExponentialSmoothing


df = pd.read_csv("../0_data/1_cleaned/fact_daily_demand.csv")

df["order_date"] = pd.to_datetime(df["order_date"])

df = df.sort_values(["product_id", "order_date"])

df.head()

,order_date,product_id,product_name,category_name,department_name,daily_demand,daily_sales,daily_profit
18780,2017-04-24,19,Nike Men's Fingertrap Max Training Shoe,Soccer,Fitness,1,112.49,54.00
18826,2017-04-26,19,Nike Men's Fingertrap Max Training Shoe,Soccer,Fitness,1,113.74,52.32
18977,2017-05-03,19,Nike Men's Fingertrap Max Training Shoe,Soccer,Fitness,1,116.24,51.15
18999,2017-05-04,19,Nike Men's Fingertrap Max Training Shoe,Soccer,Fitness,1,118.12,56.70
19042,2017-05-06,19,Nike Men's Fingertrap Max Training Shoe,Soccer,Fitness,1,118.74,-86.09


In [2]:
# 2. Forecast demand for each product


# Forecast every product separately
FORECAST_DAYS = 30

forecast_rows = []


# Loop through products
for product_id, group in df.groupby("product_id"):

    group = (
        group.sort_values("order_date")
             .set_index("order_date")
    )

    demand = group["daily_demand"]


    # Skip products with insufficient history
    if len(demand) < 14:
        continue

    try:

        model = ExponentialSmoothing(
            demand,
            trend="add",
            seasonal="add",
            seasonal_periods=7
        )

        fit = model.fit(optimized=True)

        forecast = fit.forecast(FORECAST_DAYS)

    except:
        continue


    # Calculate statistics
    avg_daily_demand = demand.mean()

    demand_std = demand.std()

    historical_30d = demand.tail(30).sum()

    last_sale_date = demand.index.max()

    forecast_total = forecast.sum()

    demand_change_pct = (
        (forecast_total - historical_30d)
        / historical_30d
    ) * 100 if historical_30d > 0 else 0


    # Demand trend
    if demand_change_pct >= 10:
        trend = "Increasing"

    elif demand_change_pct <= -10:
        trend = "Declining"

    else:
        trend = "Stable"


    # Product info
    info = group.iloc[-1]


    # Create Future Dates
    future_dates = pd.date_range(
        last_sale_date + pd.Timedelta(days=1),
        periods=FORECAST_DAYS,
        freq="D"
    )


    # Create forecast rows
    for date, prediction in zip(future_dates, forecast):

        forecast_rows.append({

            "forecast_date": date,

            "product_id": product_id,

            "product_name": info["product_name"],

            "category_name": info["category_name"],

            "department_name": info["department_name"],

            "last_sale_date": last_sale_date,

            "avg_daily_demand": round(avg_daily_demand,2),

            "historical_30d_demand": int(historical_30d),

            "forecast_demand": max(round(prediction),0),

            "forecast_30d_total": round(forecast_total),

            "demand_change_pct": round(demand_change_pct,2),

            "demand_trend": trend,

            "forecast_method": "Holt-Winters"

        })

c:\Users\Denis\OneDrive\Documents\Work\Data Analyst\Portfolio\Inventory Optimization & Demand Forecasting Analysis\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\Denis\OneDrive\Documents\Work\Data Analyst\Portfolio\Inventory Optimization & Demand Forecasting Analysis\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
c:\Users\Denis\OneDrive\Documents\Work\Data Analyst\Portfolio\Inventory Optimization & Demand Forecasting Analysis\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
 

In [ ]:
# 3. Combine all forecasts into a single DataFrame and save to CSV


fact_forecast = pd.DataFrame(forecast_rows)

fact_forecast["forecast_demand"] = (
    fact_forecast["forecast_demand"]
    .astype(int)
)

fact_forecast["historical_30d_demand"] = (
    fact_forecast["historical_30d_demand"]
    .astype(int)
)

fact_forecast["forecast_30d_total"] = (
    fact_forecast["forecast_30d_total"]
    .astype(int)
)

fact_forecast.head()

fact_forecast.to_csv(
    "../3_outputs/fact_forecast.csv",
    index=False
)

,forecast_date,product_id,product_name,category_name,department_name,last_sale_date,avg_daily_demand,historical_30d_demand,forecast_demand,forecast_30d_total,demand_change_pct,demand_trend,forecast_method
0,2017-09-11,19,Nike Men's Fingertrap Max Training Shoe,Soccer,Fitness,2017-09-10,1.19,38,1,41,7.15,Stable,Holt-Winters
1,2017-09-12,19,Nike Men's Fingertrap Max Training Shoe,Soccer,Fitness,2017-09-10,1.19,38,1,41,7.15,Stable,Holt-Winters
2,2017-09-13,19,Nike Men's Fingertrap Max Training Shoe,Soccer,Fitness,2017-09-10,1.19,38,1,41,7.15,Stable,Holt-Winters
3,2017-09-14,19,Nike Men's Fingertrap Max Training Shoe,Soccer,Fitness,2017-09-10,1.19,38,1,41,7.15,Stable,Holt-Winters
4,2017-09-15,19,Nike Men's Fingertrap Max Training Shoe,Soccer,Fitness,2017-09-10,1.19,38,1,41,7.15,Stable,Holt-Winters
